# LTX 2.5 Pro

<p align="center">
  <img src="../../.github/images/banners/fal-ltx-2.5-pro.png" alt="LTX 2.5 Pro: product-page loops that hold detail, text, and camera moves" width="900" />
</p>

High-fidelity text-to-video and image-to-video with camera-motion presets, frame-rate control, and synchronized audio, from Lightricks through fal on Microsoft Foundry.

| | |
|---|---|
| Released | 2026-11-06 (preview) |
| Publisher | fal (model by Lightricks) |
| Capability | [Video Generation](../../3-primers/video-generation.md) |
| Model card | [Fal-LTX-2.5-Pro](https://ai.azure.com/catalog/models/Fal-LTX-2.5-Pro) |
| Announcement | [Release post](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/building-the-next-generation-of-multimodal-ai-experiences-with-fal-in-microsoft-/4570032) |
| Docs | [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) |
| Deployment | Global Standard, pay-as-you-go |
| Modes | Text-to-video, image-to-video (start and optional end frame) |
| Clip shape | 6, 8, 10 s or `auto` · 720p, 1080p · 24, 25, 50 fps · 16:9, 9:16 · audio on by default |
| Rate limit | 2 requests per minute on quota tiers 1–4 |
| Pricing | Not on the model card yet. Check your deployment in the Foundry portal; fal's direct price is on [fal.ai](https://fal.ai/models/fal-ai/ltx-2.5/text-to-video/pro) |
| Env vars | `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_LTX_25_PRO_DEPLOYMENT`, `DEMO_MODE` |

**Read first:** the [Video Generation primer](../../3-primers/video-generation.md) explains the terms we use here (camera motion, frame rate, temporal consistency, native audio). Ten minutes there makes this notebook much easier.

**First time here?** Run [setup.ipynb](../../setup.ipynb) once to create your Microsoft Foundry project and `.env`.

> *"Can LTX 2.5 Pro give every product page a short loop that looks like our product, and keeps looking like it?"*

Contoso Outdoors wants a 6-second loop at the top of each tent's product page: muted, sharp, consistent across products, and true to the catalog. LTX 2.5 Pro is the quality tier of the LTX family. Its fast sibling, [LTX 2.5 Fast](../fal-ltx-2.5-fast/), trades some fidelity for speed and longer clips. We use the [Contoso Outdoors catalog](../../2-scenarios/contoso-retail/) and launch images made by an earlier MAI workshop.

## What We Will Do

- Build product-page loops straight from catalog data, and hold one look across three tents
- Direct the camera with presets, and test whether on-screen text survives animation
- See when frame rate and `auto` duration matter
- Measure every clip, and package what we learn for a customer conversation

> Run each cell with **Shift + Enter**. Guess first: before each video, predict one thing it will get right and one it will get wrong.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Before you start

**Just reading?** Leave `DEMO_MODE=replay`. The notebook shows clips saved in `examples/` and makes no Azure calls.

**Running it live?** We need four things:

1. **fal enabled on the subscription.** An admin registers the **Fal.EnableDeploy** preview feature ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#enable-fal-on-foundry)). It can take 30 minutes.
2. **An LTX 2.5 Pro deployment** (Global Standard) in our Foundry project ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#deploy-fal-models-from-the-foundry-portal)).
3. **A sign-in with a Foundry role** on the resource: run `az login` in a terminal. We use keyless Microsoft Entra ID tokens, never API keys.
4. **`.env` at the repo root** with `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_LTX_25_PRO_DEPLOYMENT`, and `DEMO_MODE=live`. See [sample.env](../../1-scripts/sample.env).

**Data handling.** fal hosts this model outside Microsoft. Prompts and images we send leave Microsoft, and Microsoft's product terms and DPA don't apply. We only send fictional Contoso data here.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "1-scripts" / "fal_video.py").exists())
load_dotenv(ROOT / ".env")

DEMO_MODE = None  # Set to "live" or "replay" to override .env
MODE = (DEMO_MODE or os.getenv("DEMO_MODE") or "replay").strip().lower()
ENDPOINT = os.getenv("MICROSOFT_FOUNDRY_ENDPOINT", "").strip()
DEPLOYMENT = os.getenv("AZURE_FAL_LTX_25_PRO_DEPLOYMENT", "Fal-LTX-2.5-Pro").strip()
RPM = int(os.getenv("FAL_REQUESTS_PER_MINUTE", "2"))
PLAY_INLINE = os.getenv("PLAY_INLINE", "true").strip().lower() == "true"

if MODE not in ("replay", "live"):
    raise SystemExit(f'DEMO_MODE is "{MODE}". Use "replay" or "live".')

credential = None
if MODE == "live":
    if not ENDPOINT.startswith("https://"):
        raise SystemExit("MICROSOFT_FOUNDRY_ENDPOINT is missing from .env. Run setup.ipynb, or copy it from the project's Overview page.")
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    try:
        credential.get_token("https://cognitiveservices.azure.com/.default")
    except Exception as error:
        raise SystemExit(f"Couldn't get a Microsoft Entra token. Run `az login` in a terminal, then run this cell again.\n{error}")

print(f"Mode: {MODE} | Deployment: {DEPLOYMENT} | {RPM} requests per minute")

The shared helper in [1-scripts/fal_video.py](../../1-scripts/fal_video.py) submits each job, polls until it's done, downloads the MP4, and saves a JSON record. LTX routes end in the tier: `text-to-video/pro`, `image-to-video/pro`. We also load the catalog, because our prompts come from real product data.

In [ ]:
import json
import re

sys.path.insert(0, str(ROOT / "1-scripts"))
from fal_video import FalVideo, as_url, scoreboard

HERE = ROOT / "4-models" / "fal-ltx-2.5-pro"
DATA = ROOT / "2-scenarios" / "contoso-retail" / "data"
IMAGES = DATA / "generated" / "images"

fal = FalVideo(ENDPOINT, DEPLOYMENT, family="lightricks/ltx-2.5", here=HERE, mode=MODE,
               requests_per_minute=RPM, play_inline=PLAY_INLINE, credential=credential)
clips = []

products = {p["slug"]: p for p in json.loads((DATA / "products.json").read_text(encoding="utf-8"))}


def spec(slug: str, field: str) -> str:
    """Read one spec (Color, Capacity, ...) from a product manual."""
    manual = (DATA / products[slug]["manual"]).read_text(encoding="utf-8")
    match = re.search(rf"\*\*{field}\*?\*?:?\*?\*?\s*:?\s*(.+)", manual)
    return match.group(1).strip() if match else "unknown"


for slug in ("trailmaster-x4-tent", "alpine-explorer-tent", "skyview-2-person-tent"):
    print(f"{products[slug]['name']:24} {spec(slug, 'Color'):8} {spec(slug, 'Capacity')}")

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 1. What does a product-page loop look like?

Product pages autoplay **muted**, so we turn audio off (`generate_audio=False`) and keep the camera calm. 720p and 6 seconds are enough for a page header and keep render time down.

> 🔮 **Guess first:** will the tent be green, as the catalog says?

In [ ]:
def loop_prompt(slug: str) -> str:
    p = products[slug]
    return (f"Product video of the {p['name']} by {p['brand']}, a {spec(slug, 'Capacity')} {spec(slug, 'Color').lower()} tent, "
            "pitched on a grassy meadow at golden hour. A gentle breeze moves the fabric; the tent stays taut. "
            "Clean, calm, premium outdoor-brand look. No people, no text.")

print(loop_prompt("trailmaster-x4-tent"))
loop = fal.run("01-product-loop", "text-to-video/pro", prompt=loop_prompt("trailmaster-x4-tent"),
               duration=6, resolution="720p", aspect_ratio="16:9", generate_audio=False)
clips.append(loop)
fal.show(loop)

**What to look for:** the color (the manual says green, but the workshop images show a green body with an orange rainfly; which did the model pick?), the tent's shape staying fixed while the fabric moves, and a clean loop point. Text-only prompts get *a* green tent, not *our* tent. Section 3 fixes that.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 2. Can one treatment hold across three tents?

A product line needs a **consistent look**: same light, same framing, same pace. We send the same template, filled from the catalog, for three tents.

In [ ]:
lineup = []
for slug in ("alpine-explorer-tent", "skyview-2-person-tent"):
    lineup.append(fal.run(f"02-lineup-{slug}", "text-to-video/pro", prompt=loop_prompt(slug),
                          duration=6, resolution="720p", aspect_ratio="16:9", generate_audio=False))
clips += lineup
fal.show(loop, *lineup)

**What to look for:** set the three strips side by side. Is the *style* consistent (light, meadow, framing) while each *product* differs in color and size? An 8-person tent should look bigger than a 2-person one. The catalog's numbers only help if the model shows them.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 3. Do the camera presets do what they say?

LTX takes `camera_motion` as a setting, not just prompt text: `dolly_in`, `dolly_out`, `dolly_left`, `dolly_right`, `jib_up`, `jib_down`, `static`, `focus_shift`. We start from the workshop's tent image (image-to-video, so the product is ours) and try three moves.

In [ ]:
from IPython.display import Image as Picture, display

display(Picture(filename=str(IMAGES / "prop-tent.webp"), width=260))

MOVES = ("dolly_in", "jib_up", "focus_shift")
moves = [fal.run(f"03-camera-{move}", "image-to-video/pro", image_url=as_url(IMAGES / "prop-tent.webp"),
                 prompt="The tent stands in a quiet studio; the fabric shifts slightly as if in a light breeze.",
                 camera_motion=move, duration=6, resolution="720p", generate_audio=False)
         for move in MOVES]
clips += moves
fal.show(*moves)

**What to look for:** match each strip to its move. `dolly_in` should grow the tent, `jib_up` should rise and look down, `focus_shift` should pull focus between foreground and background. Presets are more repeatable than describing the move in words, which matters when a whole product line needs the same shot.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 4. Does on-screen text survive animation?

The fal docs say LTX Pro decodes video in a way that keeps faces sharp and **on-screen text legible**. We test that with the workshop's launch poster: a headline, a product name, and a $250 price badge. Here 1080p is worth it.

In [ ]:
display(Picture(filename=str(IMAGES / "launch-poster.webp"), width=260))

poster = fal.run("04-poster-text", "image-to-video/pro", image_url=as_url(IMAGES / "launch-poster.webp"),
                 prompt="Mist drifts slowly across the lake behind the tent; the sunrise light brightens. All text stays exactly as it is.",
                 camera_motion="static", duration=6, resolution="1080p", aspect_ratio="9:16", generate_audio=False)
clips.append(poster)
fal.show(poster, frames=6)

**What to look for:** read "HOME, ANYWHERE." and "$250" in the **last** frame, not the first. Price and brand text must not change. If any letter drifts, the clip can't ship. That rule applies to every video model.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 5. When does frame rate matter?

LTX Pro renders at 24, 25, or 50 fps. Slow scenes look the same at any of them; **fast motion**, like a flickering flame, is where 50 fps shows. Sound is on this time: a stove clip is half its sound.

> 🔮 **Guess first:** will the 50 fps clip take twice as long to make?

In [ ]:
STOVE = (f"Close-up of the {products['ecofire-camping-stove']['name']}, a {spec('ecofire-camping-stove', 'Color').lower()} single-burner camping stove, "
         "lighting with a whoosh; blue flames flicker and a pot of water starts to steam. Static camera, shallow depth of field. "
         "Audio: the click of the igniter, the hiss of the flame, bubbling water.")

flame_25 = fal.run("05-flame-25fps", "text-to-video/pro", prompt=STOVE, fps=25, duration=6, resolution="720p")
flame_50 = fal.run("05-flame-50fps", "text-to-video/pro", prompt=STOVE, fps=50, duration=6, resolution="720p")
clips += [flame_25, flame_50]
fal.show(flame_25, flame_50)

**What to look for:** play both and watch the flame tips. Then check the facts lines: the fps should read 25 and 50, the 50 fps file should be larger, and the render time may differ. Pick 50 fps when motion is the point, 24 or 25 when the look is "cinematic".

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 6. Should we let the model choose the length?

`duration="auto"` lets LTX pick a length that suits the prompt. A simple action may come back short; a sequence may come back long. We compare a fixed 10 seconds with `auto` for a short sequence.

In [ ]:
SEQUENCE = ("A hiker unrolls the TrailMaster X4 tent on a meadow, clicks the poles together, raises the dome, "
            "and stakes the last corner as the sun sets. Handheld, documentary style. Audio: fabric, pole clicks, birds.")

fixed = fal.run("06-duration-10", "text-to-video/pro", prompt=SEQUENCE, duration=10, resolution="720p")
auto = fal.run("06-duration-auto", "text-to-video/pro", prompt=SEQUENCE, duration="auto", resolution="720p")
clips += [fixed, auto]
fal.show(fixed, auto, frames=8)

**What to look for:** the delivered seconds for `auto`, and whether the fixed 10-second clip rushes or pads the action. `auto` is handy for exploring; fixed durations are better when a slot (a 6-second ad, a 10-second loop) is already defined.

### 🔧 Break it on purpose

LTX 2.5 **Fast** renders up to 2160p. What does **Pro** say if we ask for 2160p?

In [ ]:
if MODE == "live":
    try:
        fal.run("break-2160p", "text-to-video/pro", prompt=STOVE, duration=6, resolution="2160p")
    except RuntimeError as error:
        print(error)
else:
    print("Replay mode: in live mode the service answers 422 for `resolution`, listing the allowed values 720p and 1080p.")

The same family, two tiers, different limits: Pro tops out at **1080p and 10 seconds**; Fast goes to **2160p and 20 seconds**. The error message is the quickest way to learn each tier's limits, and nothing is billed because the request never reaches the queue.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 7. What did each clip cost us in time?

### Scoreboard

Every clip we made, from its saved record: how long the job took end to end, the duration we asked for vs the one we got, and the output's shape and size.

In [ ]:
scoreboard(clips)

**What to look for:** how render time grows with resolution (the 1080p poster) and frame rate (the 50 fps flame), what `auto` actually chose, and the file sizes for a web page. For pricing, use the deployment's page in the Foundry portal, not estimates.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Teach it to a customer

### What it does, and why it matters

- **Quality-tier video, 6–10 s, up to 1080p**, with synchronized audio we can switch off.
- **Camera presets** (`dolly_in`, `jib_up`, `focus_shift`, ...) make the same shot repeatable across a product line.
- **Legible text and sharp detail**: worth testing for anything with a price, label, or logo.
- **Frame-rate control** (24, 25, 50 fps) for cinematic vs smooth motion.

### A prompt recipe that works

`[Product name and brand], [fixed facts from the catalog: color, size], [setting and light]. [One gentle action]. [Style]. [No people / no text, when true].`

Fill the template from catalog data so every product gets the same treatment. Use image-to-video when the product must look exactly like ours.

### Honest answers to common questions

- **Is it production-ready?** It's in public preview on Foundry. Validate quality and terms before production.
- **Where does our data go?** To fal, outside Microsoft. Microsoft's product terms, DPA, and certifications don't apply. Review [fal's terms](https://fal.ai/legal) with the customer.
- **Pro or Fast?** Pro for hero assets and text; [Fast](../fal-ltx-2.5-fast/) for drafts, volume, longer clips, and 4K.
- **How fast can we scale?** Quota tiers 1–4 allow 2 requests per minute; tiers 5–7 allow 40.

### When something goes wrong

- **422 on `resolution` or `duration`** → Pro allows 720p/1080p and 6/8/10 s or `auto` → use Fast for more.
- **422 on `fps` with 48** → Pro offers 24, 25, 50 → pick one of those.
- **The product doesn't look like ours** → switch to image-to-video from a product photo.
- **Text drifts** → keep the camera `static`, shorten the clip, and tell the model the text must stay unchanged.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Your Turn to Explore

- **Start and end.** Use `campsite-before.webp` as `image_url` and `campsite-after.webp` as `end_image_url` for a colorway reveal.
- **Go vertical.** Re-run section 2 at `aspect_ratio="9:16"` for a phone version of the line-up.
- **Beyond tents.** Write a loop template for backpacks and fill it from the catalog.

In [ ]:
# Your experiment here. Try: fal.run("my-clip", "text-to-video/pro", prompt="...", duration=6, resolution="720p")

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## What We Did

- ✅ Built product-page loops from catalog data and held one look across three tents
- ✅ Directed the camera with presets and tested whether on-screen text survives animation
- ✅ Saw when frame rate and `auto` duration matter
- ✅ Measured every clip and packaged the findings for a customer conversation

➡️ **Next:** [LTX 2.5 Fast](../fal-ltx-2.5-fast/) makes the same kind of clip faster, longer, and up to 4K.

Stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

**Learn more:** [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) · [Video Generation primer](../../3-primers/video-generation.md) · [LTX 2.5 Pro API on fal](https://fal.ai/models/fal-ai/ltx-2.5/text-to-video/pro/api)